                                       End-to-End PySpark Project
                                   Movie Ratings Analytics Platform

Project Title - Scalable Movie Ratings Analytics Pipeline Using PySpark



Project Overview

Business Problem

A movie streaming company collects millions of user ratings daily.
The analytics team wants to:

- Analyse user behaviour
- Identify trending/popular movies
- Calculate average ratings
- Detect low-quality or corrupt records
- Generate reporting datasets for dashboards
- Handle incremental incoming data efficiently

As a Data Engineer, your responsibility is to build a production-grade PySpark ETL pipeline that:

- Reads raw ratings data
- Cleans and validates records
- Handles null and duplicate values
- Joins business reference tables
- Performs aggregations
- Generates analytical outputs
- Stores processed data efficiently


 Dataset Details

Ratings Dataset
Column	Description
user_id	Unique user ID
movie_id	Unique movie ID
rating	Rating from 1-5
timestamp	Rating timestamp


Movie Details Dataset

Column	Description
movie_id	Unique movie ID
movie_name	Movie name
genre	Genre
release_year	Release year


In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("Movie Ratings ETL Pipeline") \
    .getOrCreate()

In [0]:
ratings_df = spark.read.option("header", True).csv("/Volumes/workspace/default/movies_dataset/ratings.csv", inferSchema=True)
movies_df = spark.read.option("header", True).csv("/Volumes/workspace/default/movies_dataset/movies.csv", inferSchema=True)

In [0]:
display(ratings_df)

user_id,movie_id,rating,timestamp
1,101,5,1700000000
1,102,4,1700000500
2,101,3,1700000600
2,103,2,1700000700
3,104,5,1700000800
3,105,null,1700000900
4,106,4,1700001000
4,101,5,1700001100
5,102,1,1700001200
5,999,3,1700001300


In [0]:
display(movies_df)

movie_id,movie_name,genre,release_year
101,Inception,Sci-Fi,2010
102,Titanic,Romance,1997
103,Interstellar,Sci-Fi,2014
104,The Dark Knight,Action,2008
105,Avengers: Endgame,Action,2019
106,The Notebook,Romance,2004


In [0]:
# Removing null ratings

ratings_clean = ratings_df.filter(col("rating").isNotNull())
display(ratings_clean)

user_id,movie_id,rating,timestamp
1,101,5,1700000000
1,102,4,1700000500
2,101,3,1700000600
2,103,2,1700000700
3,104,5,1700000800
4,106,4,1700001000
4,101,5,1700001100
5,102,1,1700001200
5,999,3,1700001300


In [0]:
# Remove duplicates
 
ratings_clean = ratings_clean.dropDuplicates(["user_id", "movie_id", "timestamp"])
display(ratings_clean)

user_id,movie_id,rating,timestamp
1,101,5,1700000000
4,106,4,1700001000
5,999,3,1700001300
1,102,4,1700000500
4,101,5,1700001100
5,102,1,1700001200
3,104,5,1700000800
2,101,3,1700000600
2,103,2,1700000700


In [0]:
# Filter invalid ratings

ratings_clean = ratings_clean.filter((col("rating") >= 1) & (col("rating") <= 5))
display(ratings_clean)

user_id,movie_id,rating,timestamp
1,101,5,1700000000
4,106,4,1700001000
5,999,3,1700001300
1,102,4,1700000500
4,101,5,1700001100
5,102,1,1700001200
3,104,5,1700000800
2,101,3,1700000600
2,103,2,1700000700


In [0]:
# Join movie Details
ratings_enriched = ratings_clean.join(movies_df, on="movie_id", how="inner")
display(ratings_enriched)

movie_id,user_id,rating,timestamp,movie_name,genre,release_year
101,1,5,1700000000,Inception,Sci-Fi,2010
106,4,4,1700001000,The Notebook,Romance,2004
102,1,4,1700000500,Titanic,Romance,1997
101,4,5,1700001100,Inception,Sci-Fi,2010
102,5,1,1700001200,Titanic,Romance,1997
104,3,5,1700000800,The Dark Knight,Action,2008
101,2,3,1700000600,Inception,Sci-Fi,2010
103,2,2,1700000700,Interstellar,Sci-Fi,2014


In [0]:
#  convert timestamp
ratings_enriched = ratings_enriched.withColumn(
    "rating_time",
    from_unixtime(col("timestamp")))
display(ratings_enriched)    

movie_id,user_id,rating,timestamp,movie_name,genre,release_year,rating_time
101,1,5,1700000000,Inception,Sci-Fi,2010,2023-11-14 22:13:20
106,4,4,1700001000,The Notebook,Romance,2004,2023-11-14 22:30:00
102,1,4,1700000500,Titanic,Romance,1997,2023-11-14 22:21:40
101,4,5,1700001100,Inception,Sci-Fi,2010,2023-11-14 22:31:40
102,5,1,1700001200,Titanic,Romance,1997,2023-11-14 22:33:20
104,3,5,1700000800,The Dark Knight,Action,2008,2023-11-14 22:26:40
101,2,3,1700000600,Inception,Sci-Fi,2010,2023-11-14 22:23:20
103,2,2,1700000700,Interstellar,Sci-Fi,2014,2023-11-14 22:25:00


In [0]:
# Extract year/month
ratings_enriched = ratings_enriched.withColumn("year", year(col("rating_time"))) \
                                   .withColumn("month", month(col("rating_time")))
display(ratings_enriched)

movie_id,user_id,rating,timestamp,movie_name,genre,release_year,rating_time,year,month
101,1,5,1700000000,Inception,Sci-Fi,2010,2023-11-14 22:13:20,2023,11
106,4,4,1700001000,The Notebook,Romance,2004,2023-11-14 22:30:00,2023,11
102,1,4,1700000500,Titanic,Romance,1997,2023-11-14 22:21:40,2023,11
101,4,5,1700001100,Inception,Sci-Fi,2010,2023-11-14 22:31:40,2023,11
102,5,1,1700001200,Titanic,Romance,1997,2023-11-14 22:33:20,2023,11
104,3,5,1700000800,The Dark Knight,Action,2008,2023-11-14 22:26:40,2023,11
101,2,3,1700000600,Inception,Sci-Fi,2010,2023-11-14 22:23:20,2023,11
103,2,2,1700000700,Interstellar,Sci-Fi,2014,2023-11-14 22:25:00,2023,11


In [0]:
# Average rating per movie
movie_avg_rating = ratings_enriched.groupBy("movie_id", "movie_name") \
    .agg(avg("rating").alias("avg_rating"),count("rating").alias("total_ratings")) \
    .orderBy(desc("avg_rating"))
display(movie_avg_rating)    

movie_id,movie_name,avg_rating,total_ratings
104,The Dark Knight,5.0,1
101,Inception,4.333333333333333,3
106,The Notebook,4.0,1
102,Titanic,2.5,2
103,Interstellar,2.0,1


In [0]:
# Tending movies

trending_movies = ratings_enriched.groupBy("movie_id", "movie_name") \
    .agg(count("*").alias("recent_ratings")) \
    .orderBy(desc("recent_ratings"))
display(trending_movies)

movie_id,movie_name,recent_ratings
101,Inception,3
102,Titanic,2
106,The Notebook,1
104,The Dark Knight,1
103,Interstellar,1


In [0]:
movie_avg_rating.write.mode("overwrite").parquet("/Volumes/workspace/default/movies_dataset/movie_avg_rating")
trending_movies.write.mode("overwrite").parquet("/Volumes/workspace/default/movies_dataset/trending_movies")
ratings_enriched.write.mode("overwrite").parquet("/Volumes/workspace/default/movies_dataset/clean_ratings")